# U06 练习 | 词向量与 RNN 基础

**本练习目标**:

- 理解 Embedding 的查表机制
- 掌握 RNN 的输入输出格式
- 实现基于 RNN 的序列分类

**通关条件**: 6.1 ~ 6.4 全部通过

## 练习 6.1: Embedding 查表

验证 `nn.Embedding` 的本质就是"查表"——给定词索引, 取出对应行。

**要求**:
1. 创建 `vocab_size=20, embed_dim=4` 的 Embedding 层
2. 打印初始权重矩阵
3. 用索引 `[1, 5, 10]` 取向量
4. 手动用权重矩阵的 fancy indexing 验证结果一致

**验证**: `embedding(idx)` 的第 0 行应该等于 `embedding.weight[1]`。

In [2]:
import torch
import torch.nn as nn

vocab_size = 20
embed_dim = 4

embedding = nn.Embedding(vocab_size, embed_dim)

# TODO 1: 打印 embedding.weight (查看完整词表)
# print(embedding.weight)

# TODO 2: 用 embedding 取索引 [1, 5, 10] 的向量
idx = torch.tensor([1, 5, 10])
vecs_by_lookup = embedding(idx)  # TODO

# TODO 3: 用 fancy indexing 直接从 weight 取对应行
vecs_by_indexing = embedding.weight[idx]  # TODO

# TODO 4: 验证两者完全相等
print(torch.allclose(vecs_by_lookup, vecs_by_indexing))   # 应该 True

True


## 练习 6.2: RNN 的输入输出

验证 RNN 的输入输出形状, 理解 `output` 和 `hn` 的区别。

**要求**:
1. 创建 `nn.RNN(input_size=8, hidden_size=16, batch_first=True)`
2. 输入形状 `(batch=3, seq_len=5, input_size=8)` 的随机 tensor
3. 初始化 `h0` 为全零
4. 前向传播, 打印 output 和 hn 的 shape
5. 验证 `hn[0]` 等于 `output[:, -1, :]`

**验证**: `torch.allclose(hn[0], output[:, -1, :])` 应该为 True。

In [3]:
import torch
import torch.nn as nn

rnn = nn.RNN(input_size=8, hidden_size=16, batch_first=True)

batch, seq_len, input_size = 3, 5, 8
x = torch.randn(batch, seq_len, input_size)
h0 = torch.zeros(1, batch, 16)

output, hn = rnn(x, h0)

# TODO 1: 打印 output.shape 和 hn.shape
print(output.shape, hn.shape)
# 期望: output (3, 5, 16),  hn (1, 3, 16)

# TODO 2: 验证 hn[0] == output[:, -1, :]
print(torch.allclose(hn[0], output[:, -1, :]))   # 应该 True

torch.Size([3, 5, 16]) torch.Size([1, 3, 16])
True


## 练习 6.3: Embedding + RNN 序列分类

实现一个简单的句子分类模型: 给定词索引序列, 判断句子情感(正面/负面)。

**要求**:
1. 定义 `SentimentRNN`: Embedding(10000, 64) -> RNN(64, 128) -> Linear(128, 2)
2. 用 8 个样本、每个 5 个词的随机数据测试
3. 前向传播验证输出形状是 `(8, 2)`
4. 计算 CrossEntropyLoss

**验证**: logits shape = `(8, 2)`, loss 是一个标量。

In [5]:
import torch
import torch.nn as nn

class SentimentRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 2)

    def forward(self, x):
        # x: (batch, seq_len) 词索引
        vecs = self.embedding(x)
        _, hn = self.rnn(vecs)
        out = self.fc(hn.squeeze(0))
        return out

# TODO 1: 实例化模型 (vocab_size=10000, embed_dim=64, hidden_dim=128)
model = SentimentRNN(vocab_size=10000, embed_dim=64, hidden_dim=128)

# TODO 2: 造 8 个句子, 每个 5 个词 (词索引在 0-9999 之间)
x = torch.randint(0,10000, (8,5))

# TODO 3: 前向传播, 打印 logits.shape, 应为 (8, 2)
logits = model(x)
print(logits.shape)

# TODO 4: 造随机标签 (0 或 1), 计算 CrossEntropyLoss
y = torch.randint(0,2, (8,))
print(y.shape)
loss = nn.CrossEntropyLoss(logits, y)

torch.Size([8, 2])
torch.Size([8])


RuntimeError: Boolean value of Tensor with more than one value is ambiguous

## 练习 6.4: 完整训练循环

把 Embedding + RNN + 分类组合起来, 在**人工生成的序列分类数据**上训练。

**数据生成逻辑**:
- 生成 1000 个"句子", 每个句子随机 5~10 个词
- 词表分两组: 前 500 词代表类别 0, 后 500 词代表类别 1
- 句子的标签 = 前 3 个词中类别 1 的词占多数 -> label=1, 否则 0

**要求**:
1. 数据生成 (人工造)
2. Embedding(1000, 32) -> RNN(32, 64) -> Linear(64, 2)
3. 训练 20 个 epoch
4. 监控 train_loss 和 val_acc
5. **pass 条件**: `val_acc > 75%`

下面三个 code cell 分别是: 数据准备 / 模型定义 / 训练循环, 按顺序运行即可。

In [ ]:
# ===== 数据准备 =====
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

VOCAB_SIZE = 1000
EMBED_DIM = 32
HIDDEN_DIM = 64

# 词表分两组: 0~499 偏向类别 0, 500~999 偏向类别 1
def make_sentences(n):
    sentences, labels = [], []
    for _ in range(n):
        seq_len = torch.randint(5, 11, (1,)).item()
        # 50% 概率倾向类别 0 (从 0~499 取词), 否则倾向类别 1 (从 500~999 取词)
        bias = torch.randint(0, 2, (1,)).item()
        if bias == 0:
            indices = torch.randint(0, 500, (seq_len,))
        else:
            indices = torch.randint(500, 1000, (seq_len,))
        # 标签由前 3 个词决定: 多数 >= 500 -> 1
        label = int((indices[:3] >= 500).sum().item() >= 2)
        sentences.append(indices)
        labels.append(label)
    return sentences, torch.tensor(labels)

torch.manual_seed(0)
sentences_train, labels_train = make_sentences(800)
sentences_val,   labels_val   = make_sentences(200)

class SimpleDataset(Dataset):
    def __init__(self, sentences, labels):
        self.sentences = sentences
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return self.sentences[idx], self.labels[idx]

def collate_fn(batch):
    sentences, labels = zip(*batch)
    seq_len = max(s.size(0) for s in sentences)
    padded = torch.zeros(len(sentences), seq_len, dtype=torch.long)
    for i, s in enumerate(sentences):
        padded[i, :s.size(0)] = s
    return padded, torch.stack(labels)

train_loader = DataLoader(SimpleDataset(sentences_train, labels_train),
                          batch_size=32, shuffle=True,  collate_fn=collate_fn)
val_loader   = DataLoader(SimpleDataset(sentences_val,   labels_val),
                          batch_size=32, shuffle=False, collate_fn=collate_fn)

print(f'train: {len(sentences_train)} 句, val: {len(sentences_val)} 句')
print(f'标签分布 (train): 0={int((labels_train==0).sum())}, 1={int((labels_train==1).sum())}')

In [ ]:
# ===== 模型定义 =====
class SeqClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(VOCAB_SIZE, EMBED_DIM)
        self.rnn = nn.RNN(EMBED_DIM, HIDDEN_DIM, batch_first=True)
        self.fc  = nn.Linear(HIDDEN_DIM, 2)

    def forward(self, x):
        # x: (batch, seq_len) 词索引
        vecs = self.embedding(x)            # (batch, seq_len, embed_dim)
        _, hn = self.rnn(vecs)              # hn: (1, batch, hidden_dim)
        return self.fc(hn.squeeze(0))       # (batch, 2)

model = SeqClassifier()
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# 快速验证 forward 形状
for xb, yb in train_loader:
    logits = model(xb)
    print(f'输入 shape: {xb.shape}, 输出 logits shape: {logits.shape}')
    break

In [ ]:
# ===== 训练循环 =====
EPOCHS = 20
val_acc = 0.0

for epoch in range(EPOCHS):
    # --- train ---
    model.train()
    train_loss = 0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        logits = model(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    # --- eval ---
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for xb, yb in val_loader:
            preds = model(xb).argmax(dim=1)
            correct += (preds == yb).sum().item()
            total   += yb.size(0)
    val_acc = correct / total

    print(f'Epoch {epoch+1:02d} | loss: {train_loss:.4f} | val_acc: {val_acc:.2%}')

# ===== 判定 =====
assert val_acc > 0.75, f'val_acc {val_acc:.2%} < 75%'
print(f'\nPASS! val_acc = {val_acc:.2%} > 75%')